## Templates

Das Template ist bewusst auf einen einfachen Platzhalter reduziert. `template.yaml` beschreibt den Ablauf und die Eingabefelder des Backstage-Templates, während der Ordner `content` die Dateien enthält, die beim Ausführen in das neue Repository kopiert werden.

Die `README.md` dient dabei als minimaler Beispielinhalt und kann später schrittweise durch die eigentliche Lemon-Shop-Applikation, Katalogdateien und Kubernetes-Konfigurationen ersetzt werden.

    ~/lemon-shop/template/
    ├── template.yaml
    └── content/
        ├── README.md

In [ ]:
%%bash
rm -rf ~/lemon-shop/examples/template/content
mkdir ~/lemon-shop/examples/template/content
cat > ~/lemon-shop/examples/template/template.yaml <<'EOF'
apiVersion: scaffolder.backstage.io/v1beta3
kind: Template
metadata:
  name: create-lemon-shop
  namespace: default
  title: Lemon Shop erstellen
  description: Platzhalter für den Lemon Shop
  tags:
    - lemon-shop
    - python
    - kubernetes
    - ecommerce
spec:
  owner: shop-team
  type: system

  parameters:
    - title: Lemon Shop
      required:
        - systemName
        - repositoryDescription
      properties:
        systemName:
          title: Systemname
          type: string
          description: Technischer Name des Lemon-Shop-Systems
          default: lemon-shop
          pattern: '^[a-z0-9]+(?:-[a-z0-9]+)*$'
          ui:autofocus: true

        systemTitle:
          title: Anzeigename
          type: string
          default: Lemon Shop

        repositoryDescription:
          title: Repository-Beschreibung
          type: string
          default: Backstage Catalog Entities für den Lemon Shop

    - title: Repository
      required:
        - repoUrl
      properties:
        repoUrl:
          title: GitHub-Repository
          type: string
          description: Organisation und Name des zu erstellenden Repositorys
          ui:field: RepoUrlPicker
          ui:options:
            allowedHosts:
              - github.com

        repoVisibility:
          title: Sichtbarkeit
          type: string
          default: private
          enum:
            - private
            - public
            - internal
          enumNames:
            - Privat
            - Öffentlich
            - Intern

  steps:
    - id: fetch
      name: Lemon-Shop-Struktur erzeugen
      action: fetch:template
      input:
        url: ./content
        values:
          systemName: ${{ parameters.systemName }}
          systemTitle: ${{ parameters.systemTitle }}
          repositoryDescription: ${{ parameters.repositoryDescription }}
          destination: ${{ parameters.repoUrl | parseRepoUrl }}

    - id: publish
      name: Repository erstellen
      action: publish:github
      input:
        repoUrl: ${{ parameters.repoUrl }}
        description: ${{ parameters.repositoryDescription }}
        repoVisibility: ${{ parameters.repoVisibility }}
        defaultBranch: main
        protectDefaultBranch: false
        sourcePath: .

    - id: register
      name: Lemon Shop registrieren
      action: catalog:register
      input:
        repoContentsUrl: ${{ steps.publish.output.repoContentsUrl }}
        catalogInfoPath: /entities.yaml

  output:
    links:
      - title: Repository öffnen
        url: ${{ steps.publish.output.remoteUrl }}
      - title: Lemon Shop im Catalog öffnen
        icon: catalog
        entityRef: system:default/${{ parameters.systemName }}
EOF

### Template Dateien

In [ ]:
%%bash
cat > ~/lemon-shop/examples/template/content/README.md <<'EOF'
# ${{ values.systemTitle }}

${{ values.repositoryDescription }}

Dieses Repository ist ein Platzhalter für den Lemon Shop.

## Bestandteile

- System: `${{ values.systemName }}`
- Component: `shop-frontend`
- Component: `product-service`
- Component: `order-service`
- Component: `payment-service`
- Resource: `database`
- Resource: `kubernetes`

EOF